# Практична робота №6. Прототип ШІ для NPC

**Дем’яненко Микола 242 2**

Прототип продовжує ПР-1–ПР-5: **UC-04 «Відступити у разі небезпеки»**, **DP-07** (чи відступати), а DP-08/DP-09 залишаються за навігацією в архітектурі ПР-4.

- **C3** передає ознаки ситуації; **C4** (ML) оцінює загрозу, **C5 + C7** перевіряють правило `HP ≤ 25% або threat_score ≥ 0.85`, якщо відступ дозволено.
- **Джерело даних:** синтетичний набір, сформований кодом нижче, без реальних логів гри. Усі оцінки в цій роботі стосуються цього навчального набору, а не готового NPC в Unity/іншій грі.
- **Запуск:** `Kernel → Restart & Run All` у Jupyter або Google Colab. Потрібні `numpy`, `pandas`, `scikit-learn` (якщо не встановлені: `%pip install numpy pandas scikit-learn`). Жодного інтернет-доступу під час самого запуску не потрібно.


## 1. Підготовка даних

Формуємо **400 тестових ситуацій** (фіксований seed = 42), з яких 300 призначені для навчання і 100 — для незалежної тестової вибірки. Основні ознаки з ПР-2: здоров’я NPC (`Health`), позиції NPC/гравця (з них обчислюється `Distance_To_Player`), видимість (`Player_Visible`) та оточення (`Environment_Data`, чи є укриття). Додано умовну силу противника (0–1). Мітка `high_threat` створена простою прозорою формулою, **а не експертним розмічуванням реальних боїв**.


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from IPython.display import display

SEED = 42
N = 400
rng = np.random.default_rng(SEED)

hp = rng.integers(5, 101, N)
distance = rng.uniform(0, 40, N).round(2)
strength = rng.uniform(0.05, 1, N).round(3)
visible = rng.integers(0, 2, N)
cover = rng.integers(0, 2, N)

# Навчальна мітка для синтетичних прикладів (не реальна ігрова істина).
risk_formula = (0.42 * (1 - hp / 100) + 0.26 * (1 - distance / 40)
                + 0.24 * strength + 0.12 * visible - 0.06 * cover)
y = (risk_formula >= 0.55).astype(int)
X = pd.DataFrame({
    'health_pct': hp,
    'distance_m': distance,
    'enemy_strength': strength,
    'player_visible': visible,
    'cover_available': cover
})
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
print(f'Набір: {len(X)} записів; навчання: {len(X_train)}; тест: {len(X_test)}')
print(f'Класи: 0 — низька загроза ({sum(y == 0)}); 1 — висока ({sum(y == 1)})')
display(X.head(5))

Набір: 400 записів; навчання: 300; тест: 100
Класи: 0 — низька загроза (255); 1 — висока (145)


,health_pct,distance_m,enemy_strength,player_visible,cover_available
0,13,31.10,0.504,1,0
1,79,38.87,0.773,0,1
2,67,20.03,0.908,0,1
3,47,5.76,0.735,0,1
4,46,0.56,0.965,1,0


## 2. Навчання моделі та оцінка загрози

Використано `RandomForestClassifier` (100 дерев, фіксований seed). Вихід `threat_score` — оцінена моделлю частка голосів за клас високої загрози (0–1). `confidence` — найбільша з двох оцінок імовірності класів; **це не калібрована гарантія правильності рекомендації**.


In [2]:
model = RandomForestClassifier(
    n_estimators=100, max_depth=7, min_samples_leaf=3, random_state=SEED
)
model.fit(X_train, y_train)
pred_test = model.predict(X_test)
classification_accuracy = accuracy_score(y_test, pred_test)
cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
print(f'Точність класифікації на 100 відкладених синтетичних записах: {classification_accuracy:.0%}')
print('Матриця помилок [[TN, FP], [FN, TP]]:')
print(cm)
assert len(X_test) == 100

Точність класифікації на 100 відкладених синтетичних записах: 91%
Матриця помилок [[TN, FP], [FN, TP]]:
[[63  1]
 [ 8 28]]


## 3. Прототип C4 → C5/C7 (DP-07)

Вхід — словник стану NPC. Координати використовує C3 для обчислення відстані. `retreat_allowed` — правило, яке встановлює розробник (D1/C7), а не гравець. Якщо дані неповні чи некоректні — дія не виконується. Якщо модель недоступна або `confidence < 0.65`, застосовується запасне правило `HP ≤ 25%`, якщо відступ дозволено. Відповідь є **рекомендацією** для C5, не прямою командою ігровому рушію.


In [3]:
def fallback(rec, reason, allow_hp_rule):
    hp_v = rec.get('health_pct')
    permitted = rec.get('retreat_allowed')
    if (allow_hp_rule and type(permitted) is bool and permitted
        and isinstance(hp_v, (int, float)) and not isinstance(hp_v, bool)
        and np.isfinite(hp_v) and 0 <= hp_v <= 100 and hp_v <= 25):
        action = 'Відступити'
    else:
        action = 'Залишитися/укритися'
    return {
        'threat_score': None, 'confidence': None, 'recommendation': action,
        'source': 'Rule-based fallback' if allow_hp_rule else 'Безпечна зупинка',
        'reason': reason, 'decision_point': 'DP-07'
    }


def recommend(rec, fitted_model=model, model_available=True):
    required = ('health_pct', 'npc_position', 'player_position',
                'enemy_strength', 'player_visible', 'cover_available',
                'retreat_allowed')
    if any(k not in rec or rec[k] is None for k in required):
        return fallback(rec, 'відсутні обов’язкові дані', allow_hp_rule=False)
    try:
        health = float(rec['health_pct'])
        strength = float(rec['enemy_strength'])
        npc_pos = np.asarray(rec['npc_position'], dtype=float)
        player_pos = np.asarray(rec['player_position'], dtype=float)
        if npc_pos.shape != (2,) or player_pos.shape != (2,):
            raise ValueError('координати X,Y')
        dist = float(np.linalg.norm(npc_pos - player_pos))
        if not np.isfinite([health, strength, dist]).all():
            raise ValueError('NaN/inf')
        if not (0 <= health <= 100 and 0 <= strength <= 1):
            raise ValueError('поза допустимими межами')
        if not all(type(rec[k]) is bool for k in
                   ('player_visible', 'cover_available', 'retreat_allowed')):
            raise ValueError('неправильні логічні ознаки')
    except (TypeError, ValueError, OverflowError):
        return fallback(rec, 'некоректні дані', allow_hp_rule=False)

    if not model_available or fitted_model is None:
        return fallback(rec, 'модель C4 недоступна', allow_hp_rule=True)

    row = pd.DataFrame([[health, dist, strength,
                         int(rec['player_visible']), int(rec['cover_available'])]],
                       columns=X.columns)
    probs = fitted_model.predict_proba(row)[0]
    threat_score = float(probs[list(fitted_model.classes_).index(1)])
    confidence = float(max(probs))

    if confidence < 0.65:
        output = fallback(rec, 'низька впевненість (< 0.65)', allow_hp_rule=True)
        output.update(threat_score=round(threat_score, 3),
                      confidence=round(confidence, 3))
        return output

    is_retreat = (rec['retreat_allowed'] and
                  (health <= 25 or threat_score >= 0.85))
    return {
        'threat_score': round(threat_score, 3),
        'confidence': round(confidence, 3),
        'recommendation': 'Відступити' if is_retreat else 'Залишитися/укритися',
        'source': 'ML + правила C5/C7',
        'reason': 'штатна перевірка', 'decision_point': 'DP-07'
    }

example = dict(health_pct=20, npc_position=(0, 0), player_position=(5, 0),
               enemy_strength=0.95, player_visible=True, cover_available=True,
               retreat_allowed=True, current_state='Бій', event_time_s=31)
print('Приклад відповіді:', recommend(example))

Приклад відповіді: {'threat_score': 0.991, 'confidence': 0.991, 'recommendation': 'Відступити', 'source': 'ML + правила C5/C7', 'reason': 'штатна перевірка', 'decision_point': 'DP-07'}


## 4. Тестування на різних ситуаціях

Очікувані дії задаються вручну за логікою UC-04 / DP-07 із ПР-3. Тут перевіряється не ігрова анімація чи маршрут (DP-08/DP-09), а **рекомендація C4 + перевірка C5/C7**, включно з винятковими ситуаціями.


In [4]:
def scenario(name, hp, dist, strength, visible, cover, allowed, expected,
             model_available=True, position_missing=False):
    rec = dict(health_pct=hp, npc_position=(0, 0),
               player_position=None if position_missing else (dist, 0),
               enemy_strength=strength, player_visible=visible,
               cover_available=cover, retreat_allowed=allowed,
               current_state='Бій', event_time_s=31)
    return dict(name=name, input=rec, expected=expected, model_available=model_available)

cases = [
    scenario('Низьке здоров’я', 20, 5, 0.95, True, True, True, 'Відступити'),
    scenario('Високе HP, далеко', 90, 35, 0.20, False, True, True, 'Залишитися/укритися'),
    scenario('Сильний ворог близько', 55, 2, 0.99, True, False, True, 'Відступити'),
    scenario('Босу заборонено втечу', 15, 2, 0.90, True, True, False, 'Залишитися/укритися'),
    scenario('Спокійна ситуація', 78, 27, 0.12, False, True, True, 'Залишитися/укритися'),
    scenario('Граничне HP = 25%', 25, 13, 0.30, True, False, True, 'Відступити'),
    scenario('HP = 26%, ворог далеко', 26, 37, 0.10, False, True, True, 'Залишитися/укритися'),
    scenario('Сильний ворог без укриття', 45, 0.5, 0.95, True, False, True, 'Відступити'),
    scenario('Модель C4 недоступна', 20, 3, 0.85, True, True, True, 'Відступити', model_available=False),
    scenario('Немає HP', None, 3, 0.85, True, True, True, 'Залишитися/укритися'),
    scenario('Низька впевненість', 30, 35, 0.50, True, False, True, 'Залишитися/укритися'),
    scenario('Немає координат', 12, 3, 0.85, True, True, True,
             'Залишитися/укритися', position_missing=True),
]

rows = []
for c in cases:
    result = recommend(c['input'], model_available=c['model_available'])
    item = c['input']
    data_short = (f"HP={item['health_pct']}, d={None if item['player_position'] is None else item['player_position'][0]} м, "
                  f"ворог={item['enemy_strength']}, відступ={'так' if item['retreat_allowed'] else 'ні'}")
    rows.append({'Ситуація / вхідні дані': f"{c['name']}: {data_short}",
                 'Очікувано': c['expected'],
                 'Отримано': result['recommendation'],
                 'Коректно?': 'так' if result['recommendation'] == c['expected'] else 'ні',
                 'Джерело рішення': result['source'],
                 'threat_score': result['threat_score'],
                 'confidence': result['confidence']})
results = pd.DataFrame(rows)
display(results)
correct = int((results['Коректно?'] == 'так').sum())
recommendation_quality = correct / len(results)
print(f'Частка коректних рекомендацій: {correct}/{len(results)} = {recommendation_quality:.1%}')
assert len(results) >= 5
assert results['Коректно?'].eq('так').all()
assert results.loc[results['Ситуація / вхідні дані'].str.startswith('Низька впевненість'),
                   'Джерело рішення'].iloc[0] == 'Rule-based fallback'


,Ситуація / вхідні дані,Очікувано,Отримано,Коректно?,Джерело рішення,threat_score,confidence
0,"Низьке здоров’я: HP=20, d=5 м, ворог=0.95, від...",Відступити,Відступити,так,ML + правила C5/C7,0.991,0.991
1,"Високе HP, далеко: HP=90, d=35 м, ворог=0.2, в...",Залишитися/укритися,Залишитися/укритися,так,ML + правила C5/C7,0.000,1.000
2,"Сильний ворог близько: HP=55, d=2 м, ворог=0.9...",Відступити,Відступити,так,ML + правила C5/C7,0.886,0.886
3,"Босу заборонено втечу: HP=15, d=2 м, ворог=0.9...",Залишитися/укритися,Залишитися/укритися,так,ML + правила C5/C7,0.980,0.980
4,"Спокійна ситуація: HP=78, d=27 м, ворог=0.12, ...",Залишитися/укритися,Залишитися/укритися,так,ML + правила C5/C7,0.000,1.000
5,"Граничне HP = 25%: HP=25, d=13 м, ворог=0.3, в...",Відступити,Відступити,так,ML + правила C5/C7,0.863,0.863
6,"HP = 26%, ворог далеко: HP=26, d=37 м, ворог=0...",Залишитися/укритися,Залишитися/укритися,так,ML + правила C5/C7,0.061,0.939
7,"Сильний ворог без укриття: HP=45, d=0.5 м, вор...",Відступити,Відступити,так,ML + правила C5/C7,0.973,0.973
8,"Модель C4 недоступна: HP=20, d=3 м, ворог=0.85...",Відступити,Відступити,так,Rule-based fallback,NaN,NaN
9,"Немає HP: HP=None, d=3 м, ворог=0.85, відступ=так",Залишитися/укритися,Залишитися/укритися,так,Безпечна зупинка,NaN,NaN


Частка коректних рекомендацій: 12/12 = 100.0%


## 5. Як інтегрується результат

`C1 / C2 → C3 → C4 (цей прототип) → C5 + C7 [DP-07] → C6 [DP-08/DP-09] → C7 → C1 (ігровий рушій)`. Розробник/QA переглядає причини та логи через C8/D4. Нижче приклад **формату повідомлення**, а не реалізація повної гри.

**Обмеження:** точність 91% отримана на синтетичних мітках, а 12/12 — на підготовлених вручну сценаріях. Значення `confidence` не каліброване; справжня надійність потребує логів із гри та перевірки незалежною QA-вибіркою. Сам прототип не будує маршрут і не керує персонажем у рушії.


In [5]:
import json
request = example
response = recommend(request)
message_to_C5 = {
    'npc_id': 'NPC-01',
    'dp_id': response['decision_point'],
    'threat_score': response['threat_score'],
    'confidence': response['confidence'],
    'recommended_action': response['recommendation'],
    'decision_source': response['source'],
    'reason_for_audit': response['reason']
}
print('C4 → C5 (приклад обміну даними):')
print(json.dumps(message_to_C5, ensure_ascii=False, indent=2))
print('Наступні компоненти: C5/C7 — рішення, C6 — маршрут, C7 — перевірка, C1 — команда рушію; C8/D4 — логування.')

C4 → C5 (приклад обміну даними):
{
  "npc_id": "NPC-01",
  "dp_id": "DP-07",
  "threat_score": 0.991,
  "confidence": 0.991,
  "recommended_action": "Відступити",
  "decision_source": "ML + правила C5/C7",
  "reason_for_audit": "штатна перевірка"
}
Наступні компоненти: C5/C7 — рішення, C6 — маршрут, C7 — перевірка, C1 — команда рушію; C8/D4 — логування.
